In [ ]:
import mlflow
import os
from dotenv import load_dotenv
load_dotenv()

mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI"))
client = mlflow.tracking.MlflowClient()

versions = client.search_model_versions("name='salary-predictor'")
for v in versions:
    print(v.version, v.current_stage, v.run_id)

In [ ]:
def get_production_val_mae(client):
    try:
        versions = client.search_model_versions("name='salary-predictor'")
        production_versions = [v for v in versions if v.current_stage == "Production"]
        production_versions = sorted(production_versions, key=lambda v: int(v.version), reverse=True)
        previous_version = production_versions[0] if production_versions else None

        if previous_version:
            run = client.get_run(previous_version.run_id)
            previous_mae = run.data.metrics.get("val/mae")
        else:
            print(f"Warning: could not retrieve previous MAE")
            previous_mae = None
        return(previous_mae)
    except Exception as e:
        print(f"Warning: could not retrieve previous MAE: {e}")
        return None

In [ ]:
client = mlflow.tracking.MlflowClient()

In [ ]:
get_production_val_mae(client)

In [ ]:
versions = client.search_model_versions("name='salary-predictor'")

In [ ]:
production_versions = [v for v in versions if v.current_stage == "Production"]

In [ ]:
production_versions = sorted(production_versions, key=lambda v: int(v.version), reverse=True)
previous_version = production_versions[0] if production_versions else None

In [ ]:
run = client.get_run(previous_version.run_id)

In [ ]:
run.data.metrics.get('model/val_mae')

In [ ]:
import pandas as pd

df = pd.read_csv("../data/processed/feature_engineered_data.csv")

mask = df['title'].str.contains('Senior Data Scientist', case=False, na=False)
print(df[mask]['avg_salary'].describe())
print(f"Count: {mask.sum()}")

In [ ]:

from utils import load_parquet_from_blob, save_parquet_to_blob


df = load_parquet_from_blob("raw/api_raw_data.parquet")
location_cols = ['location_country', 'location_state', 'location_region', 'location_city', 'location_area', 'location_suburb']
location_df = df['location.area'].apply(
    lambda x: pd.Series((list(x) + [None] * 6)[:6] if x is not None else [None] * 6)
)
    
location_df.columns = location_cols
location_df.head()

In [ ]:
df['location.area']

In [ ]:
print(type(df['location.area'].iloc[0]))
print(df['location.area'].iloc[0])

In [ ]:
try:
     df = load_parquet_from_blob("processed/feature_engineered_data.parquet")
     descriptions = load_parquet_from_blob("raw/urls_with_descriptions.parquet")
     descriptions.rename(columns={"description": "full_description"}, inplace=True)
     print(f"Loaded {len(df)} records and {len(descriptions)} descriptions from blob storage")
except Exception as e:
    print(f"ERROR: Failed to load data from blob storage: {e}")
    raise
    
df = df.merge(
    descriptions[["redirect_url", "full_description"]],
    on="redirect_url",
    how="left",
)
df["full_description"] = df["full_description"].fillna("")
df = df[df["full_description"].str.len() > 30]
print(f"Loaded {len(df)} records.")

In [ ]:
from utils import load_parquet_from_blob, save_parquet_to_blob, find_project_root
import io
PROJECT_ROOT = find_project_root()
output_file = PROJECT_ROOT / "data/raw/urls_with_descriptions.csv"
existing = pd.read_csv(output_file)
buf = io.BytesIO()
existing.to_parquet(buf, index=False)
print(f"File size: {buf.tell() / 1024 / 1024:.1f} MB")

In [ ]:
buf = io.BytesIO()
existing.to_parquet(buf, index=False, compression='gzip')
print(f"File size: {buf.tell() / 1024 / 1024:.1f} MB")

In [ ]:
df['desc_length'] = df['description'].str.len()
print(df.nlargest(3, 'desc_length')[['description', 'desc_length']])

In [ ]:
import io
import json
import os
import tempfile
from pathlib import Path

import pandas as pd
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient

STORAGE_ACCOUNT = "salaryprdata"
CONTAINER = "pipeline-data"

def _blob_client(blob_name: str):
    credential = DefaultAzureCredential()
    url = f"https://{STORAGE_ACCOUNT}.blob.core.windows.net"
    service = BlobServiceClient(url, credential=credential)
    return service.get_blob_client(container=CONTAINER, blob=blob_name)

def blob_exists(blob_name: str) -> bool:
    return _blob_client(blob_name).exists()


def save_parquet_to_blob(df: pd.DataFrame, blob_name: str) -> None:
    buf = io.BytesIO()
    df.to_parquet(buf, index=False, compression='gzip')
    buf.seek(0)
    
    client = _blob_client(blob_name)
    chunk_size = 1024 * 1024  # 1MB
    block_list = []
    block_id = 0
    
    while True:
        chunk = buf.read(chunk_size)
        if not chunk:
            break
        bid = f"{block_id:08d}"
        client.stage_block(bid, chunk)
        block_list.append(bid)
        block_id += 1
    
    client.commit_block_list(block_list)

In [ ]:
import logging
logging.basicConfig(level=logging.DEBUG)
logging.getLogger('azure').setLevel(logging.DEBUG)
logging.getLogger('azure.storage').setLevel(logging.DEBUG)

save_parquet_to_blob(existing, "raw/urls_with_descriptions.parquet")

In [ ]:
import mlflow
import os
from dotenv import load_dotenv
load_dotenv()

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
client = mlflow.tracking.MlflowClient()
print(client.search_experiments())